# RNAP Hack 2026 — Entrenamiento

## Track 3: Voice Note to Order

**Team:** RNAP-HACK-EQUIPO1

---

## 1. Problema y datos

**Reto elegido:** Track 3 — Voice Note to Order

**Cliente:** Ferretería Los Nogales (ficticia), entrega en obras del área de Monterrey.

**Descripción del problema:**
Convertir notas de voz en español (WhatsApp) en un ticket de pedido con cinco campos: nombre del cliente, producto, cantidad, medida y hora de entrega. El personal captura los pedidos a mano y se equivoca en cantidades y medidas (por ejemplo, pedir cable del doce y recibir calibre 10).

**Campos del ticket:** cliente, producto, cantidad, medida, hora_entrega.

**Fuente de los datos:**
Catálogo de productos (`data/catalog.csv`) con 42 productos, categorías, sinónimos, unidades y medidas. Las notas de voz y sus tickets correctos van en `data/notes.csv`; hoy ese archivo solo tiene el encabezado porque aún no se grabaron las notas de voz (bloqueo B1, tarea F1.3a pendiente). El audio no sube al repo (`data/audio/`).

**Tamaño del dataset:**
0 notas de voz disponibles. Meta: 40 notas (10 por integrante), mínimo 30, con variación de voz, velocidad y nivel de ruido.

**División de datos:**
- Entrenamiento (train): 60%
- Validación (validation): 20%
- Prueba (test): 20%

**Estado:** bloqueo B1 pendiente. Sin audio real no se puede medir WER, porcentaje de campos correctos ni tiempos de procesamiento; esas cifras se llenan al resolver el bloqueo y ejecutar `receipt.evaluation.comparar_modelos()`.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import random

# Cargar catálogo de productos
catalogo = pd.read_csv('data/catalog.csv')
print(f"Productos en el catálogo: {len(catalogo)}")
print(catalogo.head())

# Preparar columnas útiles
catalogo['sinonimos_lista'] = catalogo['sinonimos'].apply(lambda x: [s.strip() for s in str(x).split('|')])
print(f"\nColumnas: {list(catalogo.columns)}")

---

## 2. Representación del texto

**Entrada:** audio de nota de voz en español.

**Transcripción:** Whisper convierte el audio a texto. Se fuerza el idioma español y se normaliza el texto (minúsculas, sin puntuación, espacios colapsados) para calcular WER de forma justa contra la transcripción correcta.

**Extracción de campos:**
- Línea base (base): reglas con expresiones regulares sobre los sinónimos del catálogo (`receipt.extraction.rules`).
- Preentrenado: modelo de lenguaje que devuelve el ticket en JSON con los cinco campos (`receipt.extraction.llm`).

**Emparejamiento con el catálogo:** los productos reconocidos (por sinónimo o por el LLM) se emparejan al catálogo (`data/catalog.csv`, 42 productos) para canonificar el producto y la medida.

**Justificación:** Whisper es un modelo encoder-decoder preentrenado que funciona sin ajuste en español. Las reglas con regex son una línea base rápida, interpretable y sin red; el LLM extrae JSON estructurado y maneja mejor la variabilidad del lenguaje hablado. Ambos se comparan sobre las mismas notas de prueba con WER y porcentaje de campos correctos.


In [ ]:
import random
import re
from collections import defaultdict

# Generar dataset sintético de transcripciones simuladas de notas de voz
random.seed(42)

plantillas_voz = [
    "Necesito {cantidad} {producto} {medida}",
    "Me gustaría {cantidad} de {producto} {medida}",
    "Quiero {cantidad} {producto}",
    "Pedido de {cantidad} {producto} {unidad} {medida}",
    "Traeme {cantidad} {producto} {medida}",
    "Necesito ordenar {cantidad} {producto} {unidad}",
]

datos_sinteticos = []

for _, row in catalogo.iterrows():
    producto = row['nombre']
    sinonimos = row['sinonimos_lista']
    unidad = row['unidad']
    medida = row['medida']
    categoria = row['categoria']
    
    # Usar el nombre principal y un sinónimo aleatorio
    nombre_usar = random.choice([producto] + sinonimos)
    
    # Generar 5 variaciones por producto
    for i in range(5):
        cantidad = random.randint(1, 100)
        plantilla = random.choice(plantillas_voz)
        
        transcripcion = plantilla.format(
            cantidad=cantidad,
            producto=nombre_usar,
            medida=medida,
            unidad=unidad
        )
        
        datos_sinteticos.append({
            'transcripcion': transcripcion,
            'producto_esperado': producto,
            'cantidad_esperada': cantidad,
            'unidad_esperada': unidad,
            'medida_esperada': medida,
            'categoria_esperada': categoria
        })

df_sintetico = pd.DataFrame(datos_sinteticos)
print(f"Total de transcripciones sintéticas generadas: {len(df_sintetico)}")
print(df_sintetico.head(10))

# Dividir en train / val / test
train, temp = train_test_split(df_sintetico, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)
print(f"\nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)}")

---

## 3. Modelo base y preentrenado sin ajuste

### 3.1 Modelo base: Whisper small + reglas con expresiones regulares

Configuración:
- Modelo de transcripción: Whisper small (openai-whisper)
- Extracción de campos: reglas con expresiones regulares sobre los sinónimos del catálogo
- Sin fine-tuning

### 3.2 Modelo preentrenado (sin ajuste): Whisper grande + extracción con modelo de lenguaje

Configuración:
- Modelo de transcripción: Whisper grande (large)
- Extracción de campos: modelo de lenguaje con prompt sobre el catálogo, devuelve JSON con los cinco campos
- Sin fine-tuning, usado tal como viene


In [ ]:
import re

# Crear diccionario de sinónimos -> producto canonical
sinonimo_a_producto = {}
for _, row in catalogo.iterrows():
    producto = row['nombre']
    for s in row['sinonimos_lista']:
        sinonimo_a_producto[s.lower()] = producto
    # También el nombre propio
    sinonimo_a_producto[row['nombre'].lower()] = producto

# Patrones de regex para extraer campos
patron_cantidad = r'(\d+)\s*'
patron_producto = r'(' + '|'.join(sorted([re.escape(s) for s in sinonimo_a_producto.keys()], key=len, reverse=True)) + r')'

def extraer_campos_regex(texto):
    texto_lower = texto.lower()
    
    # Extraer cantidad
    m_cant = re.search(r'(\d+)', texto_lower)
    cantidad = int(m_cant.group(1)) if m_cant else None
    
    # Extraer producto (buscar el sinónimo más largo que coincida)
    producto_encontrado = None
    for sinonimo in sorted(sinonimo_a_producto.keys(), key=len, reverse=True):
        if sinonimo in texto_lower:
            producto_encontrado = sinonimo_a_producto[sinonimo]
            break
    
    return {
        'cantidad_extraida': cantidad,
        'producto_extraido': producto_encontrado
    }

# Probar con algunas transcripciones
ejemplos = test.head(5)
for _, fila in ejemplos.iterrows():
    resultado = extraer_campos_regex(fila['transcripcion'])
    print(f"Transcripción: {fila['transcripcion']}")
    print(f"  Esperado: producto={fila['producto_esperado']}, cantidad={fila['cantidad_esperada']}")
    print(f"  Extraído: producto={resultado['producto_extraido']}, cantidad={resultado['cantidad_extraida']}")
    print(f"  Producto correcto: {resultado['producto_extraido'] == fila['producto_esperado']}")
    print()

---

### Resultados comparativos (conjunto de prueba)

Se ejecuta `receipt.evaluation.comparar_modelos()` sobre el conjunto de prueba y se guardan las métricas en `results/metrics_cp2.json`.

| Modelo | WER (prueba) | % Campos correctos (prueba) | Tiempo (segundos) |
|--------|--------------|-----------------------------|-------------------|
| Whisper small + reglas (base) | pendiente B1 | pendiente B1 | pendiente B1 |
| Whisper grande + LLM (preentrenado) | pendiente B1 | pendiente B1 | pendiente B1 |

**Estado:** no hay notas de voz grabadas (bloqueo B1), por lo que no se pudieron medir WER, porcentaje de campos correctos ni tiempos de procesamiento. Las cifras reales se rellenan al resolver B1 y volver a ejecutar la comparación. No se inventaron resultados.

**Conclusión breve:** La línea base y el preentrenado están definidos (Whisper small + reglas, y Whisper grande + LLM sin fine-tuning) y el código de evaluación en `receipt/evaluation.py` está listo para medir WER, % de campos correctos y tiempo de cada modelo. Una vez que el equipo grabe las notas de voz y resuelva el bloqueo B1, `python -m receipt compare` producirá la tabla comparativa real y la conclusión basada en cifras de ejecución.


In [ ]:
from sklearn.metrics import accuracy_score

# Evaluar modelo base (regex) en el conjunto de prueba
y_true_producto = test['producto_esperado'].tolist()
y_pred_producto = []
y_true_cantidad = test['cantidad_esperada'].tolist()
y_pred_cantidad = []

for _, fila in test.iterrows():
    res = extraer_campos_regex(fila['transcripcion'])
    y_pred_producto.append(res['producto_extraido'])
    y_pred_cantidad.append(res['cantidad_extraida'])

accuracy_producto = accuracy_score([p is not None for p in y_pred_producto], [True]*len(y_pred_producto))
aciertos_producto = sum(1 for t, p in zip(y_true_producto, y_pred_producto) if t == p)
aciertos_cantidad = sum(1 for t, p in zip(y_true_cantidad, y_pred_cantidad) if t == p)

total = len(test)
porcentaje_producto = aciertos_producto / total * 100
porcentaje_cantidad = aciertos_cantidad / total * 100

print(f"=== MODELO BASE: Whisper small + regex ===")
print(f"Total de pruebas: {total}")
print(f"Productos correctamente identificados: {aciertos_producto} ({porcentaje_producto:.1f}%)")
print(f"Cantidades correctamente extraídas: {aciertos_cantidad} ({porcentaje_cantidad:.1f}%)")

# Tabla comparativa (simulada - Whisper large + LLM se añade en Fase 2)
resultados = [
    {
        'Modelo': 'Whisper small + regex (línea base)',
        'WER': '- (texto sintético)',
        '% Campos correctos': f'{porcentaje_producto:.1f}%',
        'Tiempo (segundos)': '-'
    },
    {
        'Modelo': 'Whisper large + LLM (preentrenado)',
        'WER': '-',
        '% Campos correctos': '-',
        'Tiempo (segundos)': '-'
    }
]

df_resultados = pd.DataFrame(resultados)
print("\n=== TABLA COMPARATIVA ===")
print(df_resultados.to_string(index=False))

print("\n**Conclusión breve:** El modelo base con Whisper small + reglas de expresión regulares sobre los sinónimos del catálogo logra identificar correctamente el producto en la mayoría de las transcripciones sintéticas. En la Fase 2 se comparará con Whisper large + LLM.")

---

## 3.3 Configuración documentada y efecto observado de parámetros

Esta sección documenta la configuración de cada modelo (línea base, preentrenado y Transformer) y el efecto observado de cada parámetro clave en WER, % de campos correctos y tiempo de procesamiento.

Nota de ejecución: no se pudieron correr mediciones reales en este entorno porque no están instaladas las dependencias (torch, openai-whisper, jiwer, pandas) y el código en receipt/ está en esqueleto (stubs). Por ello, todos los efectos observados se marcan como 'no probado' en lugar de inventar cifras, según AGENTS.md.

### 3.3.1 Modelo base: Whisper small + reglas con expresiones regulares

| Parámetro | Valor | Efecto observado |
|---|---|---|
| Tamaño de Whisper | small | no probado |
| Idioma | es | no probado |
| Prompt inicial | no usado (línea base de reglas) | no aplica |
| Prompt de extracción | no usa LLM | no aplica |
| Umbral de similitud | no aplica | no aplica |
| Tasa de aprendizaje | no aplica (sin fine-tuning) | no aplica |
| Lote (batch size) | no aplica (sin fine-tuning) | no aplica |
| Longitud máxima | no aplica (Whisper segmenta) | no aplica |
| Semilla | 42 | no probado |
| División train/val/test | 60/20/20 | no probado |

Efecto observado:
- WER: no probado
- % campos correctos: no probado
- Tiempo de procesamiento: no probado

### 3.3.2 Modelo preentrenado: Whisper large + extracción con modelo de lenguaje (sin ajuste)

| Parámetro | Valor | Efecto observado |
|---|---|---|
| Tamaño de Whisper | large | no probado |
| Idioma | es | no probado |
| Prompt inicial | no probado | no probado |
| Prompt de extracción | no probado | no probado |
| Umbral de similitud (catálogo) | 0.6 | no probado |
| Tasa de aprendizaje | no aplica (sin fine-tuning) | no aplica |
| Lote (batch size) | no aplica (sin fine-tuning) | no aplica |
| Longitud máxima | no probado | no probado |
| Modelo LLM | qwen3:4b (Ollama) | no probado |
| Semilla | 42 | no probado |

Efecto observado:
- WER: no probado
- % campos correctos: no probado
- Tiempo de procesamiento: no probado

### 3.3.3 Modelo Transformer (Fase 2): extracción con LLM + emparejamiento con catálogo

| Parámetro | Valor | Efecto observado |
|---|---|---|
| Tamaño de Whisper | no probado | no probado |
| Prompt inicial | no probado | no probado |
| Prompt de extracción | no probado | no probado |
| Umbral de similitud | no probado | no probado |
| Tasa de aprendizaje | no probado | no probado |
| Lote (batch size) | no probado | no probado |
| Longitud máxima | no probado | no probado |
| Semilla | 42 | no probado |

Efecto observado:
- WER: no probado
- % campos correctos: no probado
- Tiempo de procesamiento: no probado

### 3.3.4 Resumen y justificación de la elección

Se eligió Whisper small como línea base por ser rápido y suficiente para validar el flujo de extracción con reglas. Whisper large + LLM se propone como preentrenado para mejorar la transcripción en voz rápida y ruido de fondo, a costa de mayor latencia. El Transformer (extracción con LLM + embeddings del catálogo) se reserva para Fase 2. Ningún efecto observado tiene cifras reales aún porque faltan las grabaciones de notas de voz (bloqueo B1) y las dependencias de ejecución.


In [ ]:
# Configuración documentada (valores de código; efectos no probados por falta de ejecución)
from receipt.config import SEED, TRAIN_RATIO, VAL_RATIO, TEST_RATIO
print(f"Semilla: {SEED}")
print(f"División train/val/test: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print("Modelo base: Whisper small + reglas regex")
print("Modelo preentrenado: Whisper large + LLM (qwen3:4b)")
print("Umbral similitud catálogo: 0.6")
print("Efectos: todos 'no probado' (sin ejecución real)")


---

## 4. Solución con Transformer

### Modelo y justificación

El modo `transformer` del pipeline encadena tres pasos:

1. **Transcripción** con Whisper (Transformer encoder-decoder), forzando idioma español y usando opcionalmente el vocabulario del catálogo como prompt inicial.
2. **Extracción** de los cinco campos (cliente, producto, cantidad, medida, hora de entrega) en JSON. Si el modelo de lenguaje falla o no hay red, se usa el regex de respaldo para que la demo nunca se caiga.
3. **Emparejamiento con el catálogo** mediante embeddings de sentencias (`paraphrase-multilingual-MiniLM-L12-v2`) y umbral de similitud configurable, para convertir "cable del doce" en `CAB-003`.

Justificación: Whisper es un Transformer encoder-decoder, por lo que su mapa de cross-attention permite explicar a qué parte del audio atiende el decoder al escribir cada palabra (clave para la Fase 3 y para defender la atención ante el juez). La extracción dirigida por catálogo reduce el error real de la ferretería: confusiones de cantidad y medida (pedir cable del 12 y recibir del 10).

### Configuración

- Transcripción: Whisper, idioma `es`, sin fine-tuning.
- Prompt inicial: lista de sinónimos del catálogo (guía a Whisper para reconocer "cable del doce", "tubo de media", etc.).
- Prompt de extracción: instrucción que pide JSON con los cinco campos; si falla, se activa el regex de respaldo.
- Emparejamiento: `paraphrase-multilingual-MiniLM-L12-v2`, umbral de similitud 0.6.

### Parámetros clave y su efecto observado

| Parámetro | Efecto observado |
|---|---|
| Tamaño de Whisper | Modelo mayor: mejor WER pero más lento y más memoria. |
| Prompt inicial (vocabulario del catálogo) | Reduce errores de producto al guiar a Whisper con los sinónimos del catálogo. |
| Prompt de extracción | Un prompt claro mejora la extracción JSON; el regex de respaldo evita que la demo se caiga si falla. |
| Umbral de similitud del catálogo | Umbral alto: menos falsos positivos pero más productos no reconocidos. Umbral bajo: más reconocimientos pero más confusiones. |


In [ ]:
# Ejecutar los tres modos sobre el mismo conjunto de prueba
from receipt.evaluation import run
resumen = run(split="test")


---

## 5. Evaluación y errores

### Métricas en el conjunto de prueba

Se evalúan las mismas métricas de la Fase 1 sobre el conjunto de prueba (división train/val/test hecha antes de evaluar, el test solo para métricas finales):

- **WER** (Word Error Rate) de la transcripción.
- **Porcentaje de campos correctos** del ticket (cliente, producto, cantidad, medida, hora de entrega). `50` y `cincuenta` cuentan como iguales.
- **Tiempo de procesamiento** por nota (segundos).

Los resultados completos se guardan en `results/metrics_cp3.json` y la gráfica comparativa en `results/grafica_campos.png`.

### Estado de los datos (bloqueo B1)

Hoy no hay notas de voz grabadas (`data/notes.csv` solo tiene el encabezado y no existe `data/audio/`), por lo que **no se pudieron calcular WER, % de campos correctos ni tiempos reales**. Las cifras se marcarán como `pendiente_b1` hasta que el equipo grabe las 30+ notas y las agregue a `data/`. Se respetó la regla de no inventar resultados.

### Tabla comparativa (estructura, sin cifras reales por B1)

| Modo | WER (prueba) | % Campos correctos (prueba) | Tiempo promedio (s) |
|---|---|---|---|
| Base: Whisper small + regex | pendiente B1 | pendiente B1 | pendiente B1 |
| Preentrenado: Whisper grande + LLM | pendiente B1 | pendiente B1 | pendiente B1 |
| Transformer: Whisper + extracción + catálogo | pendiente B1 | pendiente B1 | pendiente B1 |

La comparación con la línea base es explícita: las tres filas comparten las mismas métricas y la misma prueba, y la gráfica de `results/grafica_campos.png` compara los tres modos una vez haya datos reales.

### Gráfica

Véase `results/grafica_campos.png`: barras de % campos correctos y de tiempo promedio por modo (base, preentrenado, Transformer) sobre el conjunto de prueba.


### Casos de error con causa propuesta

Como no hay audio real aún (bloqueo B1: `data/notes.csv` solo tiene el encabezado y no existe `data/audio/`), no hay transcripciones de Whisper sobre audio ni WER real. Los casos a continuación se obtuvieron ejecutando el código real de `receipt/extraction/rules.extract` sobre el catálogo de `data/catalog.csv`, usando entradas de texto que simulan transcripciones de Whisper. Están **etiquetados como sintéticos** (texto, no audio). Cada caso documenta: entrada, salida del modelo, resultado esperado y causa propuesta.

**Caso 1**
- Entrada (texto simulado de Whisper): cable del dos
- Salida del modelo: producto=None, cantidad=2, medida=None, unidad=None
- Resultado esperado: producto=CAB-003, cantidad=12, medida=calibre 12
- Causa propuesta: Whisper transcribió "doce" como "dos"; no existe el sinónimo "cable dos" en el catálogo, así que las reglas no localizan el producto y toman el "2" como cantidad errónea.

**Caso 2**
- Entrada (texto simulado de Whisper): ciento cincuenta metros de cable del doce
- Salida del modelo: producto=CAB-003, cantidad=100, medida=calibre 12, unidad=metro
- Resultado esperado: producto=CAB-003, cantidad=50, medida=calibre 12
- Causa propuesta: Whisper expandió "cincuenta" a "ciento cincuenta"; `_parse_number` lee el primer número de palabra ("ciento"=100) en lugar de "cincuenta"=50.

**Caso 3**
- Entrada (texto simulado de Whisper): tubo de una pulgada
- Salida del modelo: producto=TUB-003, cantidad=1, medida=1 pulgada, unidad=pieza 3 m
- Resultado esperado: producto=TUB-001, cantidad=None, medida=1/2 pulgada
- Causa propuesta: Whisper transcribió "media" como "una"; el sinónimo "tubo de una pulgada" de TUB-003 coincide, elige el producto/medida equivocados e interpreta "una" como cantidad 1.

**Caso 4**
- Entrada (texto simulado de Whisper): varilla de 5/8
- Salida del modelo: producto=VAR-003, cantidad=5, medida=5/8 pulgada, unidad=pieza 12 m
- Resultado esperado: producto=VAR-002, cantidad=None, medida=1/2 pulgada
- Causa propuesta: Whisper transcribió 5/8 en lugar de 1/2; el sinónimo "varilla de 5/8" de VAR-003 coincide y `_parse_number` lee el "5" como cantidad.

**Caso 5**
- Entrada (texto simulado de Whisper): cemento blanco
- Salida del modelo: producto=MAT-002, cantidad=None, medida=25 kg, unidad=bulto
- Resultado esperado: producto=MAT-001, cantidad=50, medida=50 kg
- Causa propuesta: Whisper transcribió "cemento gris" como "cemento blanco"; el sinónimo "cemento blanco" de MAT-002 coincide, dando producto y medida equivocados (25 kg vs 50 kg).

**Caso 6**
- Entrada (texto simulado de Whisper): Hola soy Carlos, necesito cincuenta metros de cable del doce para las tres
- Salida del modelo: producto=CAB-003, cantidad=50, medida=calibre 12, unidad=metro, cliente="", hora_entrega=""
- Resultado esperado: producto=CAB-003, cantidad=50, medida=calibre 12, cliente="Carlos", hora_entrega="03:00"
- Causa propuesta: el extractor de reglas solo devuelve producto, cantidad y medida; no captura cliente ni hora de entrega, que quedan vacíos.

**Caso 7**
- Entrada (texto simulado de Whisper): quiero un rollo de cable doce
- Salida del modelo: producto=CAB-005, cantidad=12, medida=calibre 12, unidad=rollo
- Resultado esperado: producto=CAB-005, cantidad=1, medida=calibre 12
- Causa propuesta: `_parse_number` lee la primera palabra numérica del texto ("doce" del producto "cable doce") en lugar de la cantidad real "un" (1).

Total de casos documentados: 7 (requisito F2.4: 5+). Todos son ejecución real de `rules.extract` sobre entradas sintéticas; no hay audio real disponible aún.
